In [2]:
import pandas as pd
import numpy as np

# ======================
# 1. LOAD DATA
# ======================
cols = [f"A{i}" for i in range(1, 17)]

df = pd.read_csv("dataset_p1/6credit+approval/crx.data", names=cols, na_values="?", skipinitialspace=True)

print("Shape:", df.shape)
print(df.head())

# ======================
# 2. MISSING VALUES
# ======================
print("\nMissing values:\n", df.isnull().sum())

# ======================
# 3. SEPARAR TARGET
# ======================
df["target"] = df["A16"].map({"+": 1, "-": 0})
df = df.drop(columns=["A16"])

# ======================
# 4. IDENTIFICAR TIPOS
# ======================
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include=["object", "string"]).columns

print("\nNum cols:", num_cols)
print("Cat cols:", cat_cols)

# ======================
# 5. IMPUTACIÓN
# ======================

# Numéricas → mediana
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Categóricas → "Missing"
df[cat_cols] = df[cat_cols].fillna("Missing")

# ======================
# 6. REDUCIR CARDINALIDAD
# ======================
for col in cat_cols:
    counts = df[col].value_counts()
    rare = counts[counts < 10].index
    df[col] = df[col].replace(rare, "Other")

# ======================
# 7. ENCODING
# ======================
df = pd.get_dummies(df, drop_first=True)

# ======================
# 8. FEATURES / TARGET
# ======================
X = df.drop(columns=["target"])
y = df["target"]

print("\nFinal shape:", X.shape)
print("Class balance:\n", y.value_counts(normalize=True))

Shape: (690, 16)
  A1     A2     A3 A4 A5 A6 A7    A8 A9 A10  A11 A12 A13    A14  A15 A16
0  b  30.83  0.000  u  g  w  v  1.25  t   t    1   f   g  202.0    0   +
1  a  58.67  4.460  u  g  q  h  3.04  t   t    6   f   g   43.0  560   +
2  a  24.50  0.500  u  g  q  h  1.50  t   f    0   f   g  280.0  824   +
3  b  27.83  1.540  u  g  w  v  3.75  t   t    5   t   g  100.0    3   +
4  b  20.17  5.625  u  g  w  v  1.71  t   f    0   f   s  120.0    0   +

Missing values:
 A1     12
A2     12
A3      0
A4      6
A5      6
A6      9
A7      9
A8      0
A9      0
A10     0
A11     0
A12     0
A13     0
A14    13
A15     0
A16     0
dtype: int64

Num cols: Index(['A2', 'A3', 'A8', 'A11', 'A14', 'A15', 'target'], dtype='str')
Cat cols: Index(['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10', 'A12', 'A13'], dtype='str')

Final shape: (690, 34)
Class balance:
 target
0    0.555072
1    0.444928
Name: proportion, dtype: float64


In [3]:
from sklearn.preprocessing import StandardScaler

num_cols = X.select_dtypes(include=np.number).columns

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])